In [35]:
from modules.RailwaySimulationGenerator import RailwaySimulationGenerator
# Importing the global libraries
import importlib, sys, os, pandas as pd
# from dotenv import load_dotenv
import pyspark.sql.types as T
import pyspark.sql as sql
import pyspark.sql.functions as F
import numpy as np
import datetime as dt

os.environ["JAVA_TOOL_OPTIONS"] = "-Djava.security.manager=allow"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.driver.extraJavaOptions=-Djava.security.manager=allow pyspark-shell"

#Mandatory
importlib.reload(importlib)
importlib.reload(sys.modules["modules.RailwaySimulationGenerator"])
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
START_DATE : str = "2025-01-01"						# Simulation start date
START_TIME : str = "06:00:00"						# Simulation start time
NB_DAYS : int = 5									# Number of days to simulate
EDGES_MAX_SPEED : float = 27.78						# Max speed of the edges in m/s (120 km/h)
TRAIN_SPEED : float = 33.33							# Train speed in m/s (120 km/h)
DIRECTORY : str = "sumo_data"
OUTPUT_DIRECTORY : str = "new_start"

filenames = {
	"stations" : f"{DIRECTORY}/stations.csv",
	"edges" : f"{DIRECTORY}/station_to_station.csv",
	"punctuality" : f"{DIRECTORY}/punctuality/202501.csv",
    "trains" : f"{DIRECTORY}/trains.add.xml"
}

# sim_stations = [
# 	288,	# Cour-sur-Heure
# 	147, 	# Berzée
# ]

# sim_stations = [
# 	291,	# Couvin
# 	798, 	# Mariembourg
# 	961, 	# Philippeville
# ]

# sim_stations = [
#     291,
#     798,
#     961,
#     1254,
#     1207,
#     976,
#     147,
#     288,
#     133,
#     612,
# 	976
# ]

sim_stations = [
	203,	# Braine-l'Alleud
	738, 	# Lillois
	911, 	# Nivelles
]

sparkSession : sql.SparkSession = (sql.SparkSession.builder
	.appName("RailwaySimulationGenerator")
	.config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow")
	.getOrCreate()
)

rsg : RailwaySimulationGenerator = RailwaySimulationGenerator(
	stations_file = filenames["stations"],
    station_to_station_file = filenames["edges"],
    punctuality_data_file = filenames["punctuality"],
    output_dir = OUTPUT_DIRECTORY,
    train_speed = TRAIN_SPEED,
    nb_days = NB_DAYS,
    edge_max_speed = EDGES_MAX_SPEED,
    start_datetime = f"{START_DATE} {START_TIME}",
    spark = sparkSession, 
    sim_stations = sim_stations 
)

In [37]:
rsg.loadData()

Loading data...
Loading stations data...
Loading station to station data...
Loading punctuality data...
Data loaded successfully


In [38]:
rsg.filterData()

Filtering data...
Filtering stations...
Filtering edges...
Filtering punctuality data...
Data filtered successfully


In [41]:
rsg.stations_df.printSchema()
print(rsg.stations_df.count())
rsg.stations_df.show(5)

root
 |-- ID: integer (nullable = true)
 |-- Geo_x: double (nullable = true)
 |-- Geo_y: double (nullable = true)
 |-- Code_TAF_TAP: string (nullable = true)
 |-- Classification: string (nullable = true)
 |-- Symbolic_name: string (nullable = true)
 |-- Name_FR_short: string (nullable = true)
 |-- Name_FR_full: string (nullable = true)

5
+----+---------+--------+------------+------------------+-------------+-------------+---------------+
|  ID|    Geo_x|   Geo_y|Code_TAF_TAP|    Classification|Symbolic_name|Name_FR_short|   Name_FR_full|
+----+---------+--------+------------+------------------+-------------+-------------+---------------+
| 738|50.643674|4.363395|     BE00738|Stop in open track|          FOL|      Lillois|        Lillois|
|1218|50.715534|4.383454|     BE01218|Stop in open track|          FWT|     Waterloo|       Waterloo|
| 919|50.535765|4.363632|     BE00919|Stop in open track|          FXB|  Obaix-Buzet|    Obaix-Buzet|
| 911|50.600493|4.335288|     BE00911|         

In [42]:
rsg.station_to_station_df.printSchema()
print(rsg.station_to_station_df.count())
rsg.station_to_station_df.show(5)

root
 |-- Departure_station_id: integer (nullable = true)
 |-- Arrival_station_id: integer (nullable = true)
 |-- Geo_x: double (nullable = true)
 |-- Geo_y: double (nullable = true)
 |-- Distance: double (nullable = true)
 |-- Coordinates: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: double (containsNull = true)

8
+--------------------+------------------+---------+--------+--------+--------------------+
|Departure_station_id|Arrival_station_id|    Geo_x|   Geo_y|Distance|         Coordinates|
+--------------------+------------------+---------+--------+--------+--------------------+
|                 738|               203| 50.66299| 4.37181| 4.74826|[[50.684759, 4.37...|
|                1218|               203|50.700274|4.379132|3.471071|[[50.715534, 4.38...|
|                 203|               738| 50.66299| 4.37181| 4.74826|[[50.684759, 4.37...|
|                 203|              1218|50.700274|4.379132|3.471071|[[50.715534, 4.38

In [30]:
rsg.punctuality_data_df.printSchema()
print(rsg.punctuality_data_df.count())
rsg.punctuality_data_df.show(5)

root
 |-- TRAIN_NO: integer (nullable = true)
 |-- RELATION: string (nullable = true)
 |-- TRAIN_SERV: string (nullable = true)
 |-- STOPPING_PLACE_ID: integer (nullable = true)
 |-- LINE_NO_DEP: string (nullable = true)
 |-- DELAY_ARR: integer (nullable = true)
 |-- DELAY_DEP: integer (nullable = true)
 |-- RELATION_DIRECTION: string (nullable = true)
 |-- LINE_NO_ARR: string (nullable = true)
 |-- REAL_DATE_ARR: string (nullable = true)
 |-- REAL_DATE_DEP: string (nullable = true)
 |-- PLANNED_DATETIME_ARR: timestamp (nullable = true)
 |-- PLANNED_DATETIME_DEP: timestamp (nullable = true)
 |-- NEXT_STOPPING_PLACE_ID: integer (nullable = true)

3085
+--------+--------+----------+-----------------+-----------+---------+---------+--------------------+-----------+-------------+-------------+--------------------+--------------------+----------------------+
|TRAIN_NO|RELATION|TRAIN_SERV|STOPPING_PLACE_ID|LINE_NO_DEP|DELAY_ARR|DELAY_DEP|  RELATION_DIRECTION|LINE_NO_ARR|REAL_DATE_ARR|REAL_DA

In [31]:
rsg.writeNetworkFiles()

Writing network files...
Writing stations file...
Writing edges file...
Writing platforms file...
Network files written successfully


In [32]:
rsg.generateNetwork(launch=True)

Generating network file...
Network file generated successfully


In [45]:
rsg.generateTrips()

Generating trips...
Number of trips generated :  87


In [34]:
for trip in rsg.trips :
	print(trip)

[{'departure_station': 919, 'arrival_station': 911, 'sumo_time': 82020}, {'departure_station': 911, 'arrival_station': 738, 'sumo_time': 82440}, {'departure_station': 738, 'arrival_station': 203, 'sumo_time': 82860}, {'departure_station': 203, 'arrival_station': 1218, 'sumo_time': 83160}]
[{'departure_station': 911, 'arrival_station': 738, 'sumo_time': 86040}, {'departure_station': 738, 'arrival_station': 203, 'sumo_time': 86460}, {'departure_station': 203, 'arrival_station': 1218, 'sumo_time': 86760}]
[{'departure_station': 1218, 'arrival_station': 203, 'sumo_time': 87540}, {'departure_station': 203, 'arrival_station': 738, 'sumo_time': 87780}]
[{'departure_station': 911, 'arrival_station': 738, 'sumo_time': 87840}, {'departure_station': 738, 'arrival_station': 203, 'sumo_time': 88260}, {'departure_station': 203, 'arrival_station': 1218, 'sumo_time': 88560}]
[{'departure_station': 1218, 'arrival_station': 203, 'sumo_time': 91140}, {'departure_station': 203, 'arrival_station': 738, 'su

In [46]:
rsg.writeScheduleFiles()

Writing schedule files...
Writing trains file...
Writing trips file...
Writing routes file...
Schedule files written successfully


In [44]:
rsg.writeConfigurationFile()

Writing configuration file...
Configuration file written successfully


In [163]:
rsg.startSimulation(launch=False)

sumo -c new_start/config.sumocfg
